# RAG Pipeline for Scientific Papers
## Notebook 01 — PDF Extraction: Text & Images

**Repository:** `rag-paper-pipeline/notebooks/01_extraction.ipynb`

Extracts text (page-by-page) and figures (embedded rasters + vector-rendered pages) from a scientific paper PDF.  
Output feeds directly into `02_figure_description.ipynb`.

```
rag-paper-pipeline/
 ├── notebooks/          ← you are here
 │    └── 01_extraction.ipynb
 ├── data/papers/        ← place your PDF here
 └── output/             ← generated output lands here
      ├── text/
      └── images/
           ├── embedded/
           └── rendered/
```

## 0. Imports & Configuration

In [ ]:
import fitz          # PyMuPDF
import pdfplumber
import os
import json
import re
from pathlib import Path
from PIL import Image
import io

# ── Paths ─────────────────────────────────────────────────────────────────────
# Notebook lives in notebooks/ → root is one level up
ROOT_DIR   = Path("__file__").resolve().parent.parent if "__file__" in dir() else Path.cwd().parent

# Fallback: if ROOT_DIR doesn't look right, set it manually
# ROOT_DIR = Path("/Users/alexandrosangelis/VS_CODE/rag-paper-pipeline")

DATA_DIR   = ROOT_DIR / "data" / "papers"
OUTPUT_DIR = ROOT_DIR / "output"

# ── Select your paper ─────────────────────────────────────────────────────────
PAPER_NAME = "your_paper.pdf"          # ← change to your PDF filename
PDF_PATH   = DATA_DIR / PAPER_NAME

# ── Image extraction settings ─────────────────────────────────────────────────
MIN_IMAGE_SIZE = 100    # pixels — ignore tiny icons/bullets
RENDER_DPI     = 150    # DPI for full-page renders (150 = good quality)
RENDER_PAGES   = True   # set False if you only want embedded raster images

# ── Output subdirectories ─────────────────────────────────────────────────────
TEXT_DIR            = OUTPUT_DIR / "text"
IMAGES_EMBEDDED_DIR = OUTPUT_DIR / "images" / "embedded"
IMAGES_RENDERED_DIR = OUTPUT_DIR / "images" / "rendered"
METADATA_PATH       = OUTPUT_DIR / "metadata.json"

for d in [TEXT_DIR, IMAGES_EMBEDDED_DIR, IMAGES_RENDERED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"ROOT    : {ROOT_DIR}")
print(f"PDF     : {PDF_PATH}")
print(f"PDF exists: {PDF_PATH.exists()}")
print()
print("Output directories:")
for d in [TEXT_DIR, IMAGES_EMBEDDED_DIR, IMAGES_RENDERED_DIR]:
    print(f"  {d}")

> **Note:** If `PDF exists: False`, either place your PDF in `data/papers/` or update `PAPER_NAME` above.

## 1. PDF Inspection
Quick diagnostic — page count, embedded image count, metadata.

In [ ]:
def inspect_pdf(pdf_path: Path) -> dict:
    """Run a quick diagnostic on the PDF."""
    doc = fitz.open(str(pdf_path))
    info = {
        "path"              : str(pdf_path),
        "page_count"        : len(doc),
        "metadata"          : doc.metadata,
        "total_images"      : 0,
        "pages_with_images" : [],
    }
    for page in doc:
        imgs = page.get_images()
        if imgs:
            info["total_images"] += len(imgs)
            info["pages_with_images"].append(page.number + 1)  # 1-indexed
    doc.close()
    return info


info = inspect_pdf(PDF_PATH)

print(f"📄 File         : {info['path']}")
print(f"📑 Pages        : {info['page_count']}")
print(f"🖼️  Embedded imgs : {info['total_images']}")
print(f"   on pages    : {info['pages_with_images']}")
print(f"\nMetadata:")
for k, v in info['metadata'].items():
    if v:
        print(f"  {k}: {v}")

## 2. Text Extraction
Uses `pdfplumber` (layout-aware, handles multi-column papers better than pypdf).  
Each page → separate `.txt` file in `output/text/`.

In [ ]:
def clean_text(raw: str) -> str:
    """Basic cleanup: fix hyphenated line breaks, collapse blank lines."""
    if not raw:
        return ""
    # Re-join hyphenated words split across lines (common in justified text)
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', raw)
    # Collapse multiple blank lines into one
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def extract_text(pdf_path: Path, output_dir: Path) -> list:
    """Extract text page-by-page. Returns list of page records."""
    pages = []
    with pdfplumber.open(str(pdf_path)) as pdf:
        total = len(pdf.pages)
        for i, page in enumerate(pdf.pages):
            page_num = i + 1
            raw  = page.extract_text(x_tolerance=2, y_tolerance=3)
            text = clean_text(raw or "")

            txt_path = output_dir / f"page_{page_num:03d}.txt"
            txt_path.write_text(text, encoding="utf-8")

            pages.append({
                "page"       : page_num,
                "char_count" : len(text),
                "text_path"  : str(txt_path),
                "text"       : text,   # kept in memory for chunking step
            })
            print(f"  Page {page_num:3d}/{total} — {len(text):,} chars")
    return pages


print("Extracting text...")
pages = extract_text(PDF_PATH, TEXT_DIR)
print(f"\n✅ {len(pages)} pages extracted — {sum(p['char_count'] for p in pages):,} total chars")

In [ ]:
# ── Quick text preview ────────────────────────────────────────────────────────
PREVIEW_PAGE = 1   # ← change to any page number

rec = pages[PREVIEW_PAGE - 1]
print(f"=== Page {PREVIEW_PAGE} ({rec['char_count']:,} chars) ===")
print(rec['text'][:1500])
if len(rec['text']) > 1500:
    print(f"\n... [{len(rec['text']) - 1500:,} more chars]")

## 3. Image Extraction

| Strategy | What it catches | Tool |
|---|---|---|
| **Embedded raster** | JPEGs/PNGs in the PDF object tree | PyMuPDF `get_images()` |
| **Full-page render** | Vector plots (matplotlib, R, LaTeX) | PyMuPDF `get_pixmap()` at 150 DPI |

> Vector plots drawn as PDF operators are **invisible** to `get_images()` — page rasterisation captures everything.

In [ ]:
def extract_embedded_images(pdf_path: Path, output_dir: Path, min_size: int = 100) -> list:
    """
    Extract raster images from the PDF object tree.
    Skips tiny images (icons, bullets). Deduplicates by xref.
    """
    doc     = fitz.open(str(pdf_path))
    records = []
    seen    = set()

    for page in doc:
        page_num = page.number + 1
        for img in page.get_images(full=True):
            xref = img[0]
            if xref in seen:
                continue
            seen.add(xref)

            try:
                pix = fitz.Pixmap(doc, xref)
                if pix.n - pix.alpha > 3:   # convert CMYK → RGB
                    pix = fitz.Pixmap(fitz.csRGB, pix)
            except Exception as e:
                print(f"  ⚠️  Page {page_num}, xref {xref}: {e}")
                continue

            if pix.width < min_size or pix.height < min_size:
                continue

            fname = f"page_{page_num:03d}_img_{xref:04d}.png"
            fpath = output_dir / fname
            pix.save(str(fpath))

            records.append({
                "type"           : "embedded",
                "page"           : page_num,
                "xref"           : xref,
                "width"          : pix.width,
                "height"         : pix.height,
                "path"           : str(fpath),
                "description"    : None,   # filled by notebook 02
                "figure_caption" : None,
            })
            print(f"  Page {page_num:3d} | xref {xref:4d} | {pix.width}×{pix.height}px → {fname}")

    doc.close()
    return records


print("Extracting embedded raster images...")
embedded_images = extract_embedded_images(PDF_PATH, IMAGES_EMBEDDED_DIR, MIN_IMAGE_SIZE)
print(f"\n✅ {len(embedded_images)} embedded images saved")

In [ ]:
def render_pages(pdf_path: Path, output_dir: Path, dpi: int = 150) -> list:
    """
    Rasterise every page at `dpi` resolution.
    Catches vector plots (matplotlib, R, LaTeX) invisible to get_images().
    """
    doc     = fitz.open(str(pdf_path))
    mat     = fitz.Matrix(dpi / 72, dpi / 72)   # PDF native = 72 dpi
    records = []

    for page in doc:
        page_num = page.number + 1
        pix      = page.get_pixmap(matrix=mat, alpha=False)

        fname = f"page_{page_num:03d}_render.png"
        fpath = output_dir / fname
        pix.save(str(fpath))

        records.append({
            "type"           : "rendered",
            "page"           : page_num,
            "width"          : pix.width,
            "height"         : pix.height,
            "dpi"            : dpi,
            "path"           : str(fpath),
            "description"    : None,   # filled by notebook 02
            "figure_caption" : None,
        })
        print(f"  Page {page_num:3d} | {pix.width}×{pix.height}px → {fname}")

    doc.close()
    return records


if RENDER_PAGES:
    print("Rendering pages (catches vector plots)...")
    rendered_pages = render_pages(PDF_PATH, IMAGES_RENDERED_DIR, dpi=RENDER_DPI)
    print(f"\n✅ {len(rendered_pages)} page renders saved")
else:
    rendered_pages = []
    print("Page rendering skipped (RENDER_PAGES=False).")

## 4. Quick Visual Check

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def show_images(records: list, max_show: int = 4, title_prefix: str = ""):
    sample = records[:max_show]
    if not sample:
        print("No images to display.")
        return
    fig, axes = plt.subplots(1, len(sample), figsize=(5 * len(sample), 5))
    if len(sample) == 1:
        axes = [axes]
    for ax, rec in zip(axes, sample):
        img = mpimg.imread(rec['path'])
        ax.imshow(img)
        ax.set_title(f"{title_prefix}Page {rec['page']}\n{rec['width']}×{rec['height']}px", fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


if embedded_images:
    print("── Embedded raster images (first 4) ──")
    show_images(embedded_images, max_show=4, title_prefix="Embedded | ")

if rendered_pages:
    print("── Full-page renders (first 4) ──")
    show_images(rendered_pages, max_show=4, title_prefix="Render | ")

## 5. Save Metadata Index
All records saved to `output/metadata.json` — used by all subsequent notebooks.

In [ ]:
metadata = {
    "pdf_path"        : str(PDF_PATH),
    "paper_name"      : PAPER_NAME,
    "page_count"      : info["page_count"],
    "pdf_metadata"    : info["metadata"],
    "text_pages"      : [{k: v for k, v in p.items() if k != "text"} for p in pages],
    "embedded_images" : embedded_images,
    "rendered_pages"  : rendered_pages,
    "summary": {
        "text_pages_extracted"  : len(pages),
        "embedded_images_found" : len(embedded_images),
        "page_renders_saved"    : len(rendered_pages),
    }
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"Metadata saved → {METADATA_PATH}")
print()
print("══ Extraction Summary ══════════════════")
for k, v in metadata["summary"].items():
    print(f"  {k:<30}: {v}")
print("════════════════════════════════════════")
print()
print("✅ Ready for notebook 02 — figure description")

## 6. Output Tree

In [ ]:
def print_tree(root: Path, max_files: int = 5):
    for dirpath, dirs, files in os.walk(root):
        dirs.sort()
        level  = Path(dirpath).relative_to(root).parts
        indent = "  " * len(level)
        print(f"{indent}📁 {Path(dirpath).name}/")
        sub   = "  " * (len(level) + 1)
        shown = sorted(files)[:max_files]
        for f in shown:
            size = (Path(dirpath) / f).stat().st_size
            print(f"{sub}📄 {f}  ({size/1024:.1f} KB)")
        if len(files) > max_files:
            print(f"{sub}... and {len(files) - max_files} more files")

print_tree(OUTPUT_DIR)

---
## Next → `02_figure_description.ipynb`

| Notebook | Description |
|---|---|
| `02_figure_description.ipynb` | Pass each image to Claude Vision → structured description → saved in metadata |
| `03_chunking_embedding.ipynb` | Chunk text + figure descriptions → embed with `text-embedding-3-small` |
| `04_retrieval_qa.ipynb` | Query → vector search → retrieve chunk + image → Claude generates answer |